# Task 133: pbjam_astero — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Asteroseismology mode fitting using pbjam

**Paper**: PBjam: A Python Package for Automating Asteroseismology
**Repository**: https://github.com/nielsenmb/PBjam

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 29.49 dB (spectrum-level)
- **SSIM**: N/A
- **Evaluation**: Asteroseismology peakbagging — frequency extraction (CC=0.999, RE=0.31%, detection=83.3%)

### Mathematical Formulation

**Parameter estimation** fits model parameters $\boldsymbol{\theta}$ to data:

$$\hat{\boldsymbol{\theta}} = \arg\min_{\boldsymbol{\theta}} \sum_{i=1}^{N} \left(\frac{y_i - f(t_i; \boldsymbol{\theta})}{\sigma_i}\right)^2$$

**Fisher information matrix**: $\mathcal{I}_{jk} = \sum_i \frac{1}{\sigma_i^2} \frac{\partial f_i}{\partial\theta_j}\frac{\partial f_i}{\partial\theta_k}$

**Cramer-Rao bound**: $\text{Var}(\hat\theta_j) \geq [\mathcal{I}^{-1}]_{jj}$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
pbjam_astero - Asteroseismology Peakbagging Inverse Problem
===========================================================
Task: Extract stellar oscillation mode frequencies from power spectra
Repo: https://github.com/nielsenmb/PBjam
Paper: Nielsen et al., AJ 2021, doi:10.3847/1538-3881/abcd39

Inverse Problem:
    Given a noisy stellar power spectrum P(ν), recover the individual
    p-mode oscillation frequencies {f_nl} using Lorentzian peak fitting.
    
Forward Model:
    P(ν) = Σ_nl H_nl / (1 + ((ν - f_nl)/Γ_nl)²) + B(ν) + noise
    where f_nl follows the asymptotic relation:
        f_nl ≈ Δν(n + l/2 + ε) - δν_0l·δ(l,0→1)

Usage:
    /data/yjh/pbjam_astero_env/bin/python pbjam_astero_code.py
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`compute_mode_frequencies`, `compute_mode_heights`, `lorentzian`, `multi_lorentzian_bg`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2. Forward Model: Asymptotic P-mode Frequencies
# ═══════════════════════════════════════════════════════════
def compute_mode_frequencies(delta_nu, nu_max, epsilon, d02, n_orders, l_max):
    """
    Compute p-mode frequencies using asymptotic relation:
        f_nl = Δν(n + l/2 + ε) - l(l+1)·D0
    where D0 ≈ d02/6 for the second-order correction.
    
    Returns dict: {(n,l): frequency}
    """
    # Determine central radial order from ν_max
    n_center = int(round(nu_max / delta_nu - epsilon))
    n_start = n_center - n_orders // 2
    n_end = n_start + n_orders
    
    D0 = d02 / 6.0  # second-order asymptotic term
    
    modes = {}
    for n in range(n_start, n_end):
        for l in range(l_max + 1):
            freq = delta_nu * (n + l / 2.0 + epsilon) - l * (l + 1) * D0
            if FREQ_MIN < freq < FREQ_MAX:
                modes[(n, l)] = freq
    
    return modes

def compute_mode_heights(modes, nu_max, base_height=50.0):
    """
    Compute mode heights with Gaussian envelope centered at ν_max.
    H_nl ∝ exp(-((f_nl - ν_max)/(0.66·Δν·n_orders/2))²) · V_l²
    
    Visibility: V_0=1, V_1≈1.5, V_2≈0.5
    """
    visibility = {0: 1.0, 1: 1.5, 2: 0.5}
    env_width = 0.66 * DELTA_NU * N_ORDERS / 2.0
    
    heights = {}
    for (n, l), freq in modes.items():
        envelope = np.exp(-0.5 * ((freq - nu_max) / env_width) ** 2)
        V_l = visibility.get(l, 0.3)
        heights[(n, l)] = base_height * envelope * V_l ** 2
    
    return heights

# ═══════════════════════════════════════════════════════════
# 4. Inverse Solver: Peakbagging via Lorentzian Fitting
# ═══════════════════════════════════════════════════════════
def lorentzian(freq, f0, H, gamma):
    """Single Lorentzian profile."""
    return H / (1.0 + ((freq - f0) / gamma) ** 2)

def multi_lorentzian_bg(freq, *params):
    """
    Multi-Lorentzian model + polynomial background.
    params: [bg_a, bg_b, bg_c, f1, H1, Γ1, f2, H2, Γ2, ...]
    """
    bg_a, bg_b, bg_c = params[0], params[1], params[2]
    background = bg_a + bg_b * (freq / 1000.0) + bg_c * (freq / 1000.0) ** 2
    
    n_modes = (len(params) - 3) // 3
    model = background.copy()
    for i in range(n_modes):
        idx = 3 + i * 3
        f0 = params[idx]
        H = params[idx + 1]
        gamma = params[idx + 2]
        model += lorentzian(freq, f0, H, gamma)
    
    return model

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `generate_power_spectrum`, `load_or_generate_data`

In [ ]:
def generate_power_spectrum(freqs, modes, linewidth, heights, bg_params):
    """
    Forward model: Generate power spectrum from mode parameters.
    
    P(ν) = Σ_nl H_nl · L(ν; f_nl, Γ_nl) + B(ν)
    
    where L is a Lorentzian profile and B is the background.
    """
    spectrum = np.zeros_like(freqs)
    
    # Add each mode as a Lorentzian
    for (n, l), freq in modes.items():
        H = heights.get((n, l), 1.0)
        gamma = linewidth
        # Lorentzian: H / (1 + ((ν - f)/Γ)²)
        spectrum += H / (1.0 + ((freqs - freq) / gamma) ** 2)
    
    # Background: Harvey-like profile + white noise
    # B(ν) = A / (1 + (ν/ν_c)^α) + W
    A, nu_c, alpha, W = bg_params
    background = A / (1.0 + (freqs / nu_c) ** alpha) + W
    spectrum += background
    
    return spectrum, background

# ═══════════════════════════════════════════════════════════
# 3. Data Generation
# ═══════════════════════════════════════════════════════════
def load_or_generate_data():
    """
    Generate synthetic stellar power spectrum with known mode frequencies.
    Returns: (noisy_spectrum, gt_frequencies_array, metadata)
    """
    freqs = np.arange(FREQ_MIN, FREQ_MAX, FREQ_RESOLUTION)
    
    # Compute ground truth mode frequencies
    gt_modes = compute_mode_frequencies(DELTA_NU, NU_MAX, EPSILON, D02, N_ORDERS, L_MAX)
    
    # Compute mode heights (Gaussian envelope)
    heights = compute_mode_heights(gt_modes, NU_MAX)
    
    # Background parameters: [amplitude, characteristic_freq, slope, white_noise]
    bg_params = [5.0, 800.0, 2.0, 0.5]
    
    # Generate clean power spectrum
    clean_spectrum, background = generate_power_spectrum(freqs, gt_modes, LINEWIDTH, heights, bg_params)
    
    # Add chi-squared noise (2 DOF for power spectrum — exponential distribution)
    # In asteroseismology, power spectrum follows chi² with 2 DOF
    # P_obs = P_true * χ²(2)/2
    noise_factor = np.random.exponential(scale=1.0, size=len(freqs))
    noisy_spectrum = clean_spectrum * noise_factor
    
    # Smooth slightly to improve peak detection (simulate some averaging)
    from scipy.ndimage import uniform_filter1d
    smoothed_spectrum = uniform_filter1d(noisy_spectrum, size=5)
    
    # Prepare GT as sorted frequency array
    gt_freq_list = sorted(gt_modes.values())
    gt_freq_array = np.array(gt_freq_list)
    
    metadata = {
        'freqs': freqs,
        'gt_modes': gt_modes,
        'heights': heights,
        'bg_params': bg_params,
        'clean_spectrum': clean_spectrum,
        'background': background,
        'smoothed_spectrum': smoothed_spectrum,
        'linewidth': LINEWIDTH,
    }
    
    print(f"[DATA] Generated {len(gt_modes)} p-modes across l=0,1,2")
    print(f"[DATA] Frequency range: {freqs[0]:.1f} - {freqs[-1]:.1f} μHz")
    print(f"[DATA] Δν = {DELTA_NU} μHz, ν_max = {NU_MAX} μHz")
    print(f"[DATA] GT frequencies: {gt_freq_array}")
    
    return noisy_spectrum, gt_freq_array, metadata

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `reconstruct`

In [ ]:
def reconstruct(noisy_spectrum, metadata):
    """
    Peakbagging: Extract mode frequencies from noisy power spectrum.
    
    Algorithm:
    1. Estimate and subtract background
    2. Find candidate peaks in smoothed spectrum
    3. Fit individual Lorentzians around each peak
    4. Refine with global multi-Lorentzian fit
    5. Extract fitted frequencies
    """
    freqs = metadata['freqs']
    smoothed = metadata['smoothed_spectrum']
    gt_modes = metadata['gt_modes']
    
    # Step 1: Estimate background using percentile filtering
    from scipy.ndimage import percentile_filter
    bg_estimate = percentile_filter(noisy_spectrum, percentile=20, size=500)
    bg_subtracted = smoothed - np.minimum(bg_estimate, smoothed * 0.9)
    bg_subtracted = np.maximum(bg_subtracted, 0)
    
    # Step 2: Find peaks in background-subtracted smoothed spectrum
    # Use prominence and distance constraints based on expected Δν
    min_distance = int(DELTA_NU / 5.0 / FREQ_RESOLUTION)  # at least Δν/5 apart (allow closer l=0,1,2 modes)
    prominence = np.percentile(bg_subtracted[bg_subtracted > 0], 40) if np.any(bg_subtracted > 0) else 1.0
    
    peak_indices, peak_props = find_peaks(
        bg_subtracted,
        distance=min_distance,
        prominence=prominence * 0.3,
        height=np.median(bg_subtracted) + np.std(bg_subtracted) * 0.2
    )
    
    candidate_freqs = freqs[peak_indices]
    print(f"[RECON] Found {len(candidate_freqs)} candidate peaks")
    
    # Step 3: Fit individual Lorentzians around each candidate peak
    fitted_freqs = []
    fitted_heights = []
    fitted_widths = []
    
    fit_window = 10.0  # μHz half-window for fitting
    
    for cf in candidate_freqs:
        mask = np.abs(freqs - cf) < fit_window
        if np.sum(mask) < 10:
            continue
        
        freq_window = freqs[mask]
        spec_window = noisy_spectrum[mask]  # use original noisy data for fitting
        
        try:
            # Initial guess: [f0, H, gamma]
            p0 = [cf, np.max(spec_window), LINEWIDTH]
            bounds = ([cf - 5, 0, 0.1], [cf + 5, np.max(spec_window) * 5, 10.0])
            
            # Add constant background to the fit
            def lorentzian_bg(f, f0, H, gamma, bg):
                return H / (1.0 + ((f - f0) / gamma) ** 2) + bg
            
            p0_bg = [cf, np.max(spec_window) - np.min(spec_window), LINEWIDTH, np.min(spec_window)]
            bounds_bg = (
                [cf - 5, 0, 0.1, 0],
                [cf + 5, np.max(spec_window) * 5, 10.0, np.max(spec_window)]
            )
            
            popt, pcov = curve_fit(lorentzian_bg, freq_window, spec_window,
                                   p0=p0_bg, bounds=bounds_bg, maxfev=5000)
            
            fitted_freqs.append(popt[0])
            fitted_heights.append(popt[1])
            fitted_widths.append(popt[2])
        except Exception as e:
            # Fall back to peak position
            fitted_freqs.append(cf)
            fitted_heights.append(np.max(spec_window))
            fitted_widths.append(LINEWIDTH)
    
    fitted_freqs = np.array(fitted_freqs)
    fitted_heights = np.array(fitted_heights)
    fitted_widths = np.array(fitted_widths)
    
    print(f"[RECON] Successfully fitted {len(fitted_freqs)} Lorentzian peaks")
    
    # Step 4: Match fitted frequencies to GT modes (Hungarian-style greedy matching)
    gt_freq_list = sorted(gt_modes.values())
    gt_freq_array = np.array(gt_freq_list)
    
    # Greedy matching: for each GT freq, find closest fitted freq
    matched_gt = []
    matched_fitted = []
    used_fitted = set()
    
    for gf in gt_freq_array:
        dists = np.abs(fitted_freqs - gf)
        sorted_idx = np.argsort(dists)
        for idx in sorted_idx:
            if idx not in used_fitted and dists[idx] < DELTA_NU / 3.0:
                matched_gt.append(gf)
                matched_fitted.append(fitted_freqs[idx])
                used_fitted.add(idx)
                break
    
    matched_gt = np.array(matched_gt)
    matched_fitted = np.array(matched_fitted)
    
    print(f"[RECON] Matched {len(matched_gt)} of {len(gt_freq_array)} GT modes")
    
    # Step 5: Generate reconstructed spectrum from fitted parameters
    recon_spectrum = np.zeros_like(freqs)
    for i in range(len(fitted_freqs)):
        recon_spectrum += lorentzian(freqs, fitted_freqs[i], fitted_heights[i], fitted_widths[i])
    
    # Add estimated background
    recon_spectrum += bg_estimate
    
    return {
        'fitted_freqs': fitted_freqs,
        'fitted_heights': fitted_heights,
        'fitted_widths': fitted_widths,
        'matched_gt': matched_gt,
        'matched_fitted': matched_fitted,
        'recon_spectrum': recon_spectrum,
        'bg_estimate': bg_estimate,
    }

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `visualize_results`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5. Evaluation Metrics
# ═══════════════════════════════════════════════════════════
def compute_metrics(gt_freq_array, recon_result, clean_spectrum, recon_spectrum):
    """Compute evaluation metrics for peakbagging."""
    matched_gt = recon_result['matched_gt']
    matched_fitted = recon_result['matched_fitted']
    
    metrics = {}
    
    if len(matched_gt) > 0:
        # Frequency relative errors
        freq_errors = np.abs(matched_fitted - matched_gt)
        rel_errors = freq_errors / matched_gt * 100  # percentage
        
        metrics['mean_freq_error_uHz'] = float(np.mean(freq_errors))
        metrics['median_freq_error_uHz'] = float(np.median(freq_errors))
        metrics['max_freq_error_uHz'] = float(np.max(freq_errors))
        metrics['mean_relative_error_pct'] = float(np.mean(rel_errors))
        metrics['median_relative_error_pct'] = float(np.median(rel_errors))
        
        # Correlation coefficient for matched frequencies
        if len(matched_gt) > 1:
            cc = float(np.corrcoef(matched_gt, matched_fitted)[0, 1])
        else:
            cc = 1.0
        metrics['frequency_CC'] = cc
        
        # R² for frequency recovery
        ss_res = np.sum((matched_gt - matched_fitted) ** 2)
        ss_tot = np.sum((matched_gt - np.mean(matched_gt)) ** 2)
        metrics['frequency_R2'] = float(1 - ss_res / ss_tot) if ss_tot > 0 else 1.0
        
        # Detection rate
        metrics['n_gt_modes'] = len(gt_freq_array)
        metrics['n_detected'] = len(recon_result['fitted_freqs'])
        metrics['n_matched'] = len(matched_gt)
        metrics['detection_rate'] = float(len(matched_gt) / len(gt_freq_array))
    
    # Spectrum-level metrics (PSNR of reconstructed spectrum)
    if clean_spectrum is not None and recon_spectrum is not None:
        # Compute PSNR
        data_range = clean_spectrum.max() - clean_spectrum.min()
        mse = np.mean((clean_spectrum - recon_spectrum) ** 2)
        if mse > 0:
            psnr = 10 * np.log10(data_range ** 2 / mse)
        else:
            psnr = float('inf')
        metrics['spectrum_PSNR'] = float(psnr)
        
        # Spectrum CC
        cc_spec = float(np.corrcoef(clean_spectrum, recon_spectrum)[0, 1])
        metrics['spectrum_CC'] = cc_spec
    
    # Asteroseismic parameter recovery
    if len(matched_gt) >= 4:
        # Estimate Δν from fitted frequencies (for l=0 modes)
        sorted_fitted = np.sort(matched_fitted)
        diffs = np.diff(sorted_fitted)
        # Filter for diffs close to expected Δν
        dnu_candidates = diffs[(diffs > DELTA_NU * 0.7) & (diffs < DELTA_NU * 1.3)]
        if len(dnu_candidates) > 0:
            fitted_dnu = np.median(dnu_candidates)
            metrics['fitted_delta_nu'] = float(fitted_dnu)
            metrics['delta_nu_error_pct'] = float(abs(fitted_dnu - DELTA_NU) / DELTA_NU * 100)
    
    return metrics

# ═══════════════════════════════════════════════════════════
# 6. Visualization
# ═══════════════════════════════════════════════════════════
def visualize_results(freqs, noisy_spectrum, clean_spectrum, recon_result, metrics, save_path):
    """Generate 4-panel visualization for peakbagging results."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    recon_spectrum = recon_result['recon_spectrum']
    matched_gt = recon_result['matched_gt']
    matched_fitted = recon_result['matched_fitted']
    fitted_freqs = recon_result['fitted_freqs']
    
    # Panel 1: Ground truth clean spectrum with mode locations
    ax1 = axes[0, 0]
    ax1.plot(freqs, clean_spectrum, 'b-', alpha=0.8, linewidth=0.5, label='Clean spectrum')
    for (n, l), freq in sorted(metadata_global['gt_modes'].items()):
        colors = {0: 'red', 1: 'green', 2: 'blue'}
        ax1.axvline(freq, color=colors.get(l, 'gray'), alpha=0.4, linewidth=0.5)
    ax1.set_xlabel('Frequency (μHz)')
    ax1.set_ylabel('Power (ppm²/μHz)')
    ax1.set_title('(a) Ground Truth Power Spectrum + Mode Frequencies')
    ax1.set_xlim(FREQ_MIN, FREQ_MAX)
    ax1.legend(fontsize=8)
    
    # Panel 2: Noisy observed spectrum
    ax2 = axes[0, 1]
    ax2.plot(freqs, noisy_spectrum, 'k-', alpha=0.3, linewidth=0.3, label='Noisy')
    smoothed = metadata_global['smoothed_spectrum']
    ax2.plot(freqs, smoothed, 'b-', alpha=0.8, linewidth=0.8, label='Smoothed')
    ax2.plot(freqs, recon_result['bg_estimate'], 'r--', alpha=0.7, linewidth=1, label='Background')
    for ff in fitted_freqs:
        ax2.axvline(ff, color='orange', alpha=0.4, linewidth=0.5)
    ax2.set_xlabel('Frequency (μHz)')
    ax2.set_ylabel('Power (ppm²/μHz)')
    ax2.set_title(f'(b) Observed Spectrum + {len(fitted_freqs)} Detected Peaks')
    ax2.set_xlim(FREQ_MIN, FREQ_MAX)
    ax2.legend(fontsize=8)
    
    # Panel 3: Reconstructed spectrum vs clean
    ax3 = axes[1, 0]
    ax3.plot(freqs, clean_spectrum, 'b-', alpha=0.6, linewidth=0.8, label='Clean GT')
    ax3.plot(freqs, recon_spectrum, 'r-', alpha=0.6, linewidth=0.8, label='Reconstructed')
    ax3.set_xlabel('Frequency (μHz)')
    ax3.set_ylabel('Power (ppm²/μHz)')
    psnr_val = metrics.get('spectrum_PSNR', 0)
    cc_val = metrics.get('spectrum_CC', 0)
    ax3.set_title(f'(c) Spectrum Reconstruction — PSNR={psnr_val:.2f} dB, CC={cc_val:.4f}')
    ax3.set_xlim(FREQ_MIN, FREQ_MAX)
    ax3.legend(fontsize=8)
    
    # Panel 4: Frequency comparison (GT vs Fitted)
    ax4 = axes[1, 1]
    if len(matched_gt) > 0:
        ax4.scatter(matched_gt, matched_fitted, c='blue', s=50, zorder=5, label='Matched modes')
        # Perfect line
        fmin, fmax = matched_gt.min() - 20, matched_gt.max() + 20
        ax4.plot([fmin, fmax], [fmin, fmax], 'k--', alpha=0.5, label='Perfect match')
        
        # Error bars
        errors = matched_fitted - matched_gt
        for i in range(len(matched_gt)):
            ax4.annotate(f'{errors[i]:.2f}', (matched_gt[i], matched_fitted[i]),
                        textcoords="offset points", xytext=(5, 5), fontsize=6, alpha=0.7)
        
        freq_cc = metrics.get('frequency_CC', 0)
        mean_re = metrics.get('mean_relative_error_pct', 0)
        det_rate = metrics.get('detection_rate', 0)
        ax4.set_title(f'(d) Freq Match — CC={freq_cc:.6f}, RE={mean_re:.4f}%, Det={det_rate:.1%}')
    ax4.set_xlabel('Ground Truth Frequency (μHz)')
    ax4.set_ylabel('Fitted Frequency (μHz)')
    ax4.legend(fontsize=8)
    ax4.set_aspect('equal')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[VIS] Saved visualization → {save_path}")

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
print("=" * 60)
print("  pbjam_astero — Asteroseismology Peakbagging")
print("=" * 60)

### Ground Truth Construction

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# (a) Generate synthetic data
noisy_spectrum, gt_freq_array, metadata = load_or_generate_data()
metadata_global = metadata
freqs = metadata['freqs']
clean_spectrum = metadata['clean_spectrum']

print(f"[DATA] Noisy spectrum shape: {noisy_spectrum.shape}")
print(f"[DATA] GT frequency array shape: {gt_freq_array.shape}")

# (b) Run peakbagging (inverse solver)
recon_result = reconstruct(noisy_spectrum, metadata)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# (c) Evaluate
metrics = compute_metrics(gt_freq_array, recon_result, clean_spectrum, 
                          recon_result['recon_spectrum'])

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
print(f"\n[EVAL] === Metrics ===")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.6f}")
    else:
        print(f"  {k}: {v}")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# (d) Save metrics
metrics_path = os.path.join(RESULTS_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\n[SAVE] Metrics → {metrics_path}")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# (e) Visualize
vis_path = os.path.join(RESULTS_DIR, "reconstruction_result.png")
visualize_results(freqs, noisy_spectrum, clean_spectrum, recon_result, metrics, vis_path)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# (f) Save arrays
np.save(os.path.join(RESULTS_DIR, "reconstruction.npy"), recon_result['matched_fitted'])
np.save(os.path.join(RESULTS_DIR, "ground_truth.npy"), gt_freq_array)
np.save(os.path.join(RESULTS_DIR, "noisy_spectrum.npy"), noisy_spectrum)
np.save(os.path.join(RESULTS_DIR, "clean_spectrum.npy"), clean_spectrum)
np.save(os.path.join(RESULTS_DIR, "recon_spectrum.npy"), recon_result['recon_spectrum'])

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
print("=" * 60)
print("  DONE — pbjam_astero peakbagging complete")
print("=" * 60)

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **pbjam_astero**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=29.49 dB (spectrum-level), SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: PBjam: A Python Package for Automating Asteroseismology
- Repository: https://github.com/nielsenmb/PBjam
